# *Escherichia coli* and *Saccharomyces cerevisiae* co-culture

The notebook shows how to:

- build a community model of *E. coli* (iAF1260) and *S. cerevisiae* (iMM904) from the models of each organism;
- simulate the co-culture on xylose;
- search for *E. coli* reaction knockouts that make *E. coli* supply L-tyrosine to the yeast. L-tyrosine is a precursor of the flavonoid naringenin, whose pathway is not in these models.

The notebook needs MEWpy and an LP solver, and runs from the `examples` folder, which contains the models. The optimization step takes a few minutes.

In [1]:
import logging

import pandas as pd
from cobra.io import read_sbml_model

from mewpy.com import CommunityModel, exchanges
from mewpy.optimization import EA
from mewpy.optimization.evaluation import BPCY, TargetFlux
from mewpy.problems import RKOProblem

# jMetalPy logs every step of the EA at DEBUG level
logging.getLogger('jmetal').setLevel(logging.WARNING)

## Load the organism models

`iAF1260.xml` defines no objective (COBRApy warns about it), so we set its biomass reaction as the objective. The model ids (`iAF1260` and `iMM904`) become the organism suffixes in the community model.

In [2]:
sc = read_sbml_model('models/yeast/iMM904.xml.gz')
ec = read_sbml_model('models/ec/iAF1260.xml')
ec.objective = 'BIOMASS'

No objective coefficients in model. Unclear what should be optimized


## Community model

We build a COBRApy community (`flavor='cobra'`), which is solved with COBRApy's solver (GLPK here). The biomasses are merged (the default), so both organisms grow at the same rate.

In [3]:
community = CommunityModel([ec, sc], flavor='cobra', verbose=False)
cmodel = community.merged_model
community.organisms_biomass

{'iAF1260': 'BIOMASS_iAF1260', 'iMM904': 'BIOMASS_SC5_notrace_iMM904'}

## Medium

Xylose is the only external carbon source (up to 10 mmol/gDW/h). We close every community exchange for uptake, then open the inorganic nutrients. Wild-type *S. cerevisiae* does not grow on xylose, so we block the yeast's xylose uptake (iMM904 has a xylose uptake route). The community has no external acetate, so the yeast can only take up acetate secreted by *E. coli*, at most 10 mmol/gDW/h. The community tyrosine exchange is closed, so L-tyrosine can only come from the organisms.

In [4]:
for rx in cmodel.get_exchange_reactions():
    cmodel.set_reaction_bounds(rx, 0, 1000, False)

inorganic = ['ca2', 'cl', 'cobalt2', 'cu', 'fe3', 'k', 'mg2', 'mn2', 'mobd', 'nh4', 'o2', 'pi', 'so4', 'zn2']
medium = {f'EX_{m}_e': (-1000, 0) for m in inorganic}
medium['EX_xyl__D_e'] = (-10, 1000)         # xylose, the only carbon source
medium['EX_xyl__D_e_iAF1260'] = (-1000, 0)  # E. coli takes up xylose ...
medium['EX_xyl__D_e_iMM904'] = (0, 1000)    # ... the yeast does not
medium['EX_ac_e_iMM904'] = (-10, 1000)      # yeast acetate uptake
medium['EX_tyr__L_e'] = 0                   # no external tyrosine

cmodel.set_environmental_conditions(medium)

## Simulation

A pFBA simulation of the co-culture:

In [5]:
wt_res = cmodel.simulate(method='pFBA')
wt_res.find('BIOMASS|growth')

,Flux rate
Reaction ID,
BIOMASS_iAF1260,0.413666
BIOMASS_SC5_notrace_iMM904,0.413666
community_growth,0.413666


The organisms' tyrosine exchanges (positive values are secretion into the shared pool, negative values are uptake):

In [6]:
wt_res.find('EX_tyr__L_e_')

,Flux rate
Reaction ID,
EX_tyr__L_e_iAF1260,-0.057137
EX_tyr__L_e_iMM904,0.057137


FVA shows the range of the tyrosine exchanges when the community grows at 90% or more of its maximum rate:

In [7]:
cmodel.FVA(reactions=['EX_tyr__L_e_iAF1260', 'EX_tyr__L_e_iMM904'])

{'EX_tyr__L_e_iAF1260': [-0.06201572827349815, 0.4629281065356514],
 'EX_tyr__L_e_iMM904': [-0.46292810653565497, 0.06201572827349988]}

At 90% or more of the maximum growth rate, *E. coli* can secrete up to 0.463 mmol/gDW/h of L-tyrosine, or take up to 0.062. pFBA, which minimises the total flux while keeping the growth rate close to its maximum, has the yeast supply 0.057 mmol/gDW/h of tyrosine to *E. coli*.

## Optimization

The target is the tyrosine secretion of *E. coli*. We use two objectives: BPCY (community growth × tyrosine secretion) and the tyrosine secretion itself, both from pFBA.

In [8]:
TARGET = 'EX_tyr__L_e_iAF1260'
BIOMASS = community.biomass

f1 = BPCY(BIOMASS, TARGET)
f2 = TargetFlux(TARGET)

The knockout candidates are the *E. coli* reactions, except the maintenance ATP demand (`ATPM`), the ATP synthase, the exchange reactions (`EX_` prefix), and the outer membrane transport and periplasmic reactions (`tex` and `pp` suffixes). We also leave out the candidates whose knockout lowers the community growth below 0.01 /h (`is_essential_reaction`):

In [9]:
candidates = []
for rxn in cmodel.reactions:
    if not rxn.endswith('_iAF1260'):
        continue
    name = rxn[:-len('_iAF1260')]
    if name == 'ATPM' or name.startswith(('EX_', 'ATPS')) or name.endswith(('tex', 'pp')):
        continue
    candidates.append(rxn)

KO_targets = [rxn for rxn in candidates if not cmodel.is_essential_reaction(rxn)]
len(candidates), len(KO_targets)

(1213, 1050)

We search for combinations of up to two knockouts, running the EA for 10 generations with a fixed seed:

In [10]:
problem = RKOProblem(cmodel,
                     fevaluation=[f1, f2],
                     target=KO_targets,
                     candidate_max_size=2)

ea = EA(problem, max_generations=10, seed=1)
solutions = ea.run(simplify=False)

2026-11-09 23:23:46,964	INFO util.py:155 -- Outdated packages:
  ipywidgets==7.8.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-11-09 23:23:47,335	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html


2026-11-09 23:23:51,919	INFO worker.py:2038 -- Started a local Ray instance.


In [11]:
df = ea.dataframe()
df.sort_values('TargetFlux', ascending=False).head(10)

,Modification,Size,BPCY,TargetFlux
0,"{'_RPE_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,0.000000e+00,0.000000e+00
1,{'_RPE_iAF1260': 0},1,-1.363450e-18,-3.340494e-18
2,"{'_RPE_iAF1260': 0, 'NDPK3_iAF1260': 0}",2,-2.074423e-17,-5.085909e-17
3,"{'AKGDH_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.326688e-02,-3.289448e-02
4,"{'OCBT_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.327608e-02,-3.290588e-02
5,"{'_LEUTAi_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.331282e-02,-3.295139e-02
6,"{'NDPK3_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.332207e-02,-3.296283e-02
7,"{'HISTD_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.332691e-02,-3.296881e-02
8,"{'PRATPP_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.332691e-02,-3.296881e-02
9,"{'_DPR_iAF1260': 0, 'ACONTa_iAF1260': 0}",2,-1.334103e-02,-3.298628e-02


## Evaluate the best solution

We simulate the solution with the highest tyrosine exchange of *E. coli*:

In [12]:
best = max(solutions, key=lambda s: s.fitness[1])
print(best.values)
res = problem.simulate(solution=best.values, method='pFBA')
res.find('BIOMASS|growth|EX_tyr__L_e_')

{'_RPE_iAF1260': 0, 'ACONTa_iAF1260': 0}


,Flux rate
Reaction ID,
BIOMASS_iAF1260,0.403004
BIOMASS_SC5_notrace_iMM904,0.403004
community_growth,0.403004


In this run, no candidate makes *E. coli* a net tyrosine exporter. The best one (`_RPE` and `ACONTa` knockouts) brings its tyrosine exchange from the wild-type uptake (0.057 mmol/gDW/h) to 0, so `find` lists no tyrosine exchange above (it hides zero fluxes). The EA evaluated only 1000 candidates (10 generations of 100), out of more than 550,000 sets of one or two knockouts.

Such knockouts exist. For example, knocking out phosphopentomutase (`PPM_iAF1260`) alone makes *E. coli* secrete tyrosine:

In [13]:
ppm_res = cmodel.simulate(method='pFBA', constraints={'PPM_iAF1260': 0})
ppm_res.find('BIOMASS|growth|EX_tyr__L_e_')

,Flux rate
Reaction ID,
EX_tyr__L_e_iAF1260,0.041826
BIOMASS_iAF1260,0.410060
EX_tyr__L_e_iMM904,-0.041826
BIOMASS_SC5_notrace_iMM904,0.410060
community_growth,0.410060


The exchanges of *E. coli* with the shared environment in the wild-type community and with the `PPM` knockout (positive values are secretion):

In [14]:
ec_ex = pd.DataFrame({'wild type': exchanges(community, wt_res)['iAF1260'],
                      'PPM knockout': exchanges(community, ppm_res)['iAF1260']})
ec_ex[(ec_ex.abs() > 1e-6).any(axis=1)].round(4)

,wild type,PPM knockout
Metabolite,,
pro__L_e,-0.0915,0.0000
mobd_e,-0.0013,-0.0013
cl_e,-0.0020,-0.0019
thymd_e,0.0000,0.0015
glu__L_e,0.5169,2.6582
asn__L_e,0.0421,0.0417
cobalt2_e,-0.0013,-0.0013
orn_e,-0.1888,-0.1872
h2o_e,4.1846,4.7528


With the knockout, *E. coli* switches from taking up to secreting L-tyrosine and L-phenylalanine, and starts taking up 2-oxoglutarate, while the community growth drops from 0.414 to 0.410 /h.